<a href="https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6_quaternions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Domínio VI: os quaternions

## A orientação no espaço num número de quatro partes: a primeira vez em que a própria rotação não comuta

**Série:** [raiz](X0_raiz.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X0_raiz.ipynb)) · **Anterior:** [X5_dualcomplexos.ipynb](X5_dualcomplexos.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X5_dualcomplexos.ipynb))

**O domínio.** Os quaternions $q=w+x\mathbf i+y\mathbf j+z\mathbf k$, com $\mathbf i^2=\mathbf j^2=\mathbf k^2=\mathbf{ijk}=-1$, fazem pela rotação do espaço o que os complexos unitários fizeram pela do plano: o giro de $\theta$ em torno do eixo $\mathbf n$ é $q=(\cos\frac\theta2,\ \mathrm{sen}\frac\theta2\,\mathbf n)$, compor giros é multiplicar, desfazer é conjugar, e um vetor gira por $q\,\mathbf v\,q^*$. O grupo $SO(3)$ aparece como a esfera $S^3$ dos quaternions unitários, com $q$ e $-q$ representando a mesma rotação.

Três consequências atravessam o domínio:
- a **própria rotação não comuta**: girar um livro em torno de $x$ e depois de $y$ não é o mesmo que o contrário; o colchete de Lie de $\mathfrak{so}(3)$ é o produto vetorial, e ele aparece nos integradores, no erro de cone das centrais inerciais e na bola que rola;
- a **dupla cobertura** tem efeitos visíveis: uma volta de 360° não traz o quaternion de volta; um controlador que ignora isso gira o satélite 350° para corrigir 10°; uma câmera lenta vê a bola girar ao contrário;
- os **ângulos de Euler** têm singularidades (o *gimbal lock*), e os quaternions não: é por isso que a navegação, a animação e os jogos usam quaternions.

**Os exemplos físicos deste domínio**
- **Um satélite** que determina a própria orientação pelas estrelas, e nuvens de pontos 3D alinhadas pelo ICP, na etapa (b).
- **A porca de Dzhanibekov**, que capota sozinha no espaço, e o **erro de cone** das centrais inerciais, na etapa (c).
- **Pepé com um celular no capacete**: quatro filtros de atitude e o erro no corpo e no mundo, na etapa (d).
- **Cinco apêndices**: controle de atitude e *unwinding*, portas de um qubit, a bola que rola, realidade virtual e a bola de pingue-pongue com efeito.

A animação mostra a porca girando em torno do eixo intermediário, à esquerda (o momento angular, em vermelho, fica parado; a haste capota), e uma bola rolando num quadrado, vista de cima, à direita (ela volta ao ponto de partida girada).

In [1]:
# Ferramentas comuns da série (repetidas em todos os notebooks, para que cada um rode sozinho no Colab)
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 100,
                     "axes.spines.top": False, "axes.spines.right": False})

def confere(nome, obtido, esperado, tol=1e-9, relativo=False):
    """Compara duas quantidades. Imprime OK ou interrompe o notebook com uma mensagem clara."""
    a, b = np.asarray(obtido), np.asarray(esperado)
    tipo = complex if (np.iscomplexobj(a) or np.iscomplexobj(b)) else float      # aceita complexos (domínio IV)
    a, b = a.astype(tipo), b.astype(tipo)
    escala = max(np.max(np.abs(b)), 1e-300) if relativo else 1.0
    erro = float(np.max(np.abs(a - b)) / escala)
    ok = erro <= tol
    print(f"{'OK ' if ok else 'FALHOU'}  {nome}:  diferença {'relativa ' if relativo else ''}máxima = {erro:.2e}")
    assert ok, f"conferência falhou: {nome}"

def sinc(x):
    """sen(x)/x, sem perder dígitos perto de 0 (np.sinc usa sen(πx)/(πx))."""
    return np.sinc(np.asarray(x, float) / np.pi)

def qmul(a, b):
    """Produto de Hamilton de quaternions guardados como arrays (w, x, y, z); aceita pilhas (..., 4)."""
    a, b = np.asarray(a), np.asarray(b)
    w1, x1, y1, z1 = np.moveaxis(a, -1, 0); w2, x2, y2, z2 = np.moveaxis(b, -1, 0)
    return np.stack([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2,
                     w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2,
                     w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2], axis=-1)

def qconj(q):
    q = np.asarray(q); return q * np.array([1, -1, -1, -1])

def qnorm(q):
    q = np.asarray(q, float); return q / np.linalg.norm(q, axis=-1, keepdims=True)

def qrot(q, v):
    """O vetor v (3,) levado pela rotação q: a parte vetorial de q (0, v) q*."""
    return qmul(qmul(q, np.concatenate([[0.0], v])), qconj(q))[1:]

def qmat(q):
    """A matriz de rotação R(q), com R v = q v q* (q unitário)."""
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])

def mat_q(R):
    """De uma matriz de rotação para um quaternion (método de Shepperd: usa o maior pivô, sem perder dígitos)."""
    t = np.trace(R); d = np.diag(R)
    k = np.argmax([t, *d])
    if k == 0:
        w = np.sqrt(1 + t) / 2; q = [w, (R[2, 1] - R[1, 2]) / (4 * w), (R[0, 2] - R[2, 0]) / (4 * w), (R[1, 0] - R[0, 1]) / (4 * w)]
    elif k == 1:
        x = np.sqrt(1 + 2 * d[0] - t) / 2; q = [(R[2, 1] - R[1, 2]) / (4 * x), x, (R[0, 1] + R[1, 0]) / (4 * x), (R[0, 2] + R[2, 0]) / (4 * x)]
    elif k == 2:
        y = np.sqrt(1 + 2 * d[1] - t) / 2; q = [(R[0, 2] - R[2, 0]) / (4 * y), (R[0, 1] + R[1, 0]) / (4 * y), y, (R[1, 2] + R[2, 1]) / (4 * y)]
    else:
        z = np.sqrt(1 + 2 * d[2] - t) / 2; q = [(R[1, 0] - R[0, 1]) / (4 * z), (R[0, 2] + R[2, 0]) / (4 * z), (R[1, 2] + R[2, 1]) / (4 * z), z]
    q = np.array(q); return q if q[0] >= 0 else -q

def qexp(phi):
    """Exp do vetor de rotação φ = θ n: (cos(θ/2), sen(θ/2) n), com o sinc estável perto de θ = 0."""
    phi = np.asarray(phi, float); th = np.linalg.norm(phi)
    return np.concatenate([[np.cos(th / 2)], 0.5 * sinc(th / 2) * phi])

def qlog(q):
    """Log: o vetor de rotação φ de q (escolhendo o representante com w ≥ 0, isto é, |θ| ≤ π)."""
    q = np.asarray(q, float)
    if q[0] < 0: q = -q
    s = np.linalg.norm(q[1:]); th = 2 * np.arctan2(s, q[0])     # atan2: estável para ângulos pequenos e grandes
    return 2 * q[1:] / sinc(th / 2)

def slerp(q0, q1, s):
    """Interpolação esférica: q0 Exp(s Log(q0* q1)), pelo arco mais curto."""
    return qmul(q0, qexp(s * qlog(qmul(qconj(q0), q1))))

def chapeu(v):
    """A matriz antissimétrica [v]×, com [v]× u = v × u."""
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

from matplotlib import animation
from IPython.display import HTML

# a porca: corpo livre com I = diag(1, 2, 3), girando perto do eixo intermediário
I = np.array([1.0, 2.0, 3.0]); dw = lambda w: np.cross(I * w, w) / I
def campo(s): return np.concatenate([0.5 * qmul(s[:4], np.concatenate([[0.0], s[4:]])), dw(s[4:])])
s = np.concatenate([[1.0, 0, 0, 0], [0.01, 2.0, 0.01]]); h = 0.01; Hs = [s]
for _ in range(2400):
    k1 = campo(s); k2 = campo(s + h / 2 * k1); k3 = campo(s + h / 2 * k2); k4 = campo(s + h * k3)
    s = s + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4); Hs.append(s)
Hs = np.array(Hs)[::40]                                 # 61 quadros em 24 s
L = qmat(qnorm(Hs[0, :4])) @ (I * Hs[0, 4:]); L = L / np.linalg.norm(L)

# a bola: rola num quadrado de lado 0,9 r (vista de cima), marca no equador
lado = 0.9; cantos = np.array([0, lado, lado + 1j * lado, 1j * lado, 0])
cam = np.concatenate([np.linspace(cantos[k], cantos[k + 1], 16)[:-1] for k in range(4)] + [[0j]])
qb = [np.array([1.0, 0, 0, 0])]
for p0, p1 in zip(cam[:-1], cam[1:]):
    d = p1 - p0; qb.append(qmul(qexp(np.cross([0, 0, 1.0], [d.real, d.imag, 0.0])), qb[-1]))

fig = plt.figure(figsize=(7.6, 3.6), dpi=60)
a1 = fig.add_subplot(1, 2, 1, projection="3d"); a2 = fig.add_subplot(1, 2, 2)
a1.set_xlim(-1, 1); a1.set_ylim(-1, 1); a1.set_zlim(-1, 1); a1.set_box_aspect((1, 1, 1))
a1.set_xticks([]); a1.set_yticks([]); a1.set_zticks([]); a1.set_title("a porca de Dzhanibekov", fontsize=9)
a1.plot([0, L[0]], [0, L[1]], [0, L[2]], color="tab:red", lw=2)
haste, = a1.plot([], [], [], color="0.2", lw=4); barra, = a1.plot([], [], [], color="tab:blue", lw=4)
a2.set_xlim(-0.6, 1.5); a2.set_ylim(-0.6, 1.5); a2.set_aspect("equal"); a2.set_xticks([]); a2.set_yticks([]); a2.grid(False)
a2.set_title("a bola que rola (vista de cima)", fontsize=9)
a2.plot(cam.real, cam.imag, ":", color="0.5", lw=1)
bola = plt.Circle((0, 0), 0.3, fill=False, color="0.4"); a2.add_patch(bola)
marca, = a2.plot([], [], color="tab:red", lw=3); inicio = qrot(qb[0], [0.3, 0, 0])
a2.plot([0, inicio[0]], [0, inicio[1]], color="tab:red", lw=1, alpha=0.4)
plt.tight_layout()

def quadro(i):
    q = qnorm(Hs[i, :4])
    e2 = qrot(q, [0, 0.9, 0]); e1 = qrot(q, [0.5, 0, 0])
    haste.set_data_3d([-e2[0], e2[0]], [-e2[1], e2[1]], [-e2[2], e2[2]])
    barra.set_data_3d([e2[0] - e1[0], e2[0] + e1[0]], [e2[1] - e1[1], e2[1] + e1[1]], [e2[2] - e1[2], e2[2] + e1[2]])
    p = cam[i]; m = qrot(qb[i], [0.3, 0, 0])
    bola.center = (p.real, p.imag); marca.set_data([p.real, p.real + m[0]], [p.imag, p.imag + m[1]])
    return haste, barra, bola, marca

anim = animation.FuncAnimation(fig, quadro, frames=61, interval=150, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

---

## O que este domínio estabelece para a série

| ideia | domínio V (dual-complexos) | domínio VI (quaternions) | o que muda depois |
|---|---|---|---|
| **o que se representa** | a pose no plano | a orientação no espaço, $(\cos\frac\theta2,\mathrm{sen}\frac\theta2\,\mathbf n)$ | a pose no espaço (VII) |
| **compor** | não comuta (por causa da translação) | não comuta **na própria rotação** | as duas não comutatividades juntas |
| **$\boxplus$** | corpo ≠ mundo, na translação | corpo ≠ mundo já na orientação: $\mathrm{Exp}(\delta)\,q=q\,\mathrm{Exp}(R^\top\delta)$ | $\mathrm{Ad}$ $6\times6$ |
| **$\mathrm{Exp}$** | um arco de círculo | a fórmula de Rodrigues; o ângulo por $\mathrm{atan2}$ | o parafuso de Chasles |
| **colchete** | $[X_1,X_2]$ mistura giro e translação | o produto vetorial: Magnus, cone, bola que rola | $\mathfrak{se}(3)$ |
| **dupla cobertura** | $q$ e $-q$ | $q$ e $-q$, com efeitos físicos: *unwinding*, serrilhado | o mesmo, com a parte dual |
| **sistemas** | Procrustes complexo | Wahba e Horn: um **autovetor** $4\times4$ | a pose de nuvens 3D, mão-olho |
| **filtro** | erro no corpo, no mundo, nas coordenadas | MEKF (corpo), IEKF (mundo), aditivo, Mahony | o ESKF de pose; o quadrirrotor |

---

## As quatro folhas deste domínio

### (a) Álgebra: operações, propriedades e implementação
[X6a_quaternions_algebra.ipynb](X6a_quaternions_algebra.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6a_quaternions_algebra.ipynb))

O produto de Hamilton, a identidade dos quatro quadrados e as matrizes de $SU(2)$, pelo `sympy`; o teste do livro (duas ordens, dois giros de 120° diferentes); a dupla cobertura, lendo laços de matrizes (360° termina em $-1$, 720° em $+1$); o *gimbal lock* (taxas de Euler de 70 rad/s para uma rotação de 0,6 rad/s, e um erro que salta sete ordens de grandeza); o ângulo por $\mathrm{atan2}$, que o $\arccos$ perde perto de zero; SLERP com velocidade constante, NLERP com velocidade desigual e Euler com um caminho 44% mais longo; rotações uniformes de Shoemake contra ângulos de Euler sorteados (rejeitados com valor-p de $10^{-91}$); 200 rotações exatas com quaternions inteiros; a interface de domínio em $SO(3)$.

### (b) Sistemas de equações lineares e não lineares
[X6b_quaternions_sistemas.ipynb](X6b_quaternions_sistemas.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6b_quaternions_sistemas.ipynb))

O problema de Wahba para um satélite: o método q de Davenport (um autovetor $4\times4$) igual ao SVD, o TRIAD, e a covariância de Shuster conferida por Monte Carlo a menos de 3%, com o eixo da linha de visada das estrelas oito vezes pior que os outros; a orientação absoluta de Horn e o ICP em 3D, que acerta sempre com chutes de até 60° e erra quase sempre a partir de 120°; a média de rotações, em que somar componentes erra 36° e as médias de Markley e de Karcher erram 2,5°.

### (c) Cinemática e dinâmica: equações diferenciais
[X6c_quaternions_edo.ipynb](X6c_quaternions_edo.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6c_quaternions_edo.ipynb))

Euler, RK4 nas componentes, Magnus 2 e Magnus 4 com e sem o colchete (ordens 1, 4, 2, 4 e 2); a porca de Dzhanibekov, que capota a cada 11 s com energia e momento angular no mundo conservados a $10^{-10}$; o erro de cone de uma central inercial, em que o algoritmo ingênuo deriva 960 °/h a 100 Hz e a correção de Miller, que é um colchete de Lie, reduz a deriva 50 vezes e sobe a ordem de 2 para 4.

### (d) Perseguição com filtro de Kalman
[X6d_quaternions_perseguicao.ipynb](X6d_quaternions_perseguicao.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6d_quaternions_perseguicao.ipynb))

Pepé, com um celular no capacete, estima a orientação da cabeça com giroscópio, acelerômetro e magnetômetro. Partindo perto da verdade, o EKF aditivo, o MEKF (erro no corpo) e o IEKF (erro no mundo) empatam (0,78°, NEES 2,75), e o filtro de Mahony erra 2,1°. Ligando o celular sem saber o rumo, o IEKF converge mais rápido (9,6° depois de 1 s, contra 20° e 24°), mas nenhum dos três EKFs é consistente no começo: a lição é inicializar pelo problema de Wahba.

---

## Apêndices

Aplicações que não cabiam nas quatro etapas, cada uma num notebook independente, com as ferramentas do domínio (quaternions, $\mathrm{Exp}$ e $\mathrm{Log}$, o colchete, filtros de atitude) e as próprias conferências.

- **O satélite que dá a volta à toa.** [X6_ap_unwinding.ipynb](X6_ap_unwinding.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6_ap_unwinding.ipynb)). Um PD de atitude que ignora o sinal de $q$ gira o satélite 350° para corrigir 10°; escolher o sinal resolve, mas faz o torque chavear dezenas de vezes perto de 180° com ruído; a histerese elimina o chaveamento.
- **As portas de um qubit são quaternions.** [X6_ap_qubit.ipynb](X6_ap_qubit.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6_ap_qubit.ipynb)). O dicionário com as matrizes de Pauli, as portas X, Y, Z, H, S e T como rotações, a esfera de Bloch, a decomposição ZYZ, e a aproximação de rotações quaisquer por palavras em H e T, com o erro caindo como $N^{-1/3}$.
- **A bola que rola.** [X6_ap_bola.ipynb](X6_ap_bola.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6_ap_bola.ipynb)). Rolar uma bola num caminho fechado pequeno a deixa girada de $-A/r^2$ em torno da vertical, qualquer que seja a forma do caminho: o colchete de Lie que se sente com a mão. Em caminhos grandes, o eixo do giro se inclina.
- **Óculos de realidade virtual.** [X6_ap_vr.ipynb](X6_ap_vr.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6_ap_vr.ipynb)). Prever a orientação da cabeça 20 ms à frente: sem previsão, 1,8° de erro; com a velocidade do giroscópio e a $\mathrm{Exp}$, 0,19°; com um filtro de Kalman de aceleração, 0,064°.
- **A bola de pingue-pongue com efeito.** [X6_ap_pingpong.ipynb](X6_ap_pingpong.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X6_ap_pingpong.ipynb)). A força de Magnus em três saques; medir a rotação pela marca da bola, e o serrilhado em $SO(3)$: abaixo de dois quadros por rotação, um *topspin* de 50 rps aparece como corte de 25 rps, e o quique previsto erra 70 cm.

---

## Para quem vem da série Kalman

Pontes com a série Kalman (pasta `kalman/` do mesmo repositório):
- o MEKF e o IEKF da etapa (d) são o ESKF e o IEKF do [notebook 5](../../kalman/notebooks/kalman_notas_im_kf_ekf_eskf_iekf_ieskf.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/kalman/notebooks/kalman_notas_im_kf_ekf_eskf_iekf_ieskf.ipynb)), agora em $SO(3)$, com o viés do giroscópio no estado;
- a etapa (d) é a parte de atitude do quadrirrotor do [notebook 6](../../kalman/notebooks/kalman_quadrotor_quaternions.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/kalman/notebooks/kalman_quadrotor_quaternions.ipynb)): o mesmo estado de erro de 3 componentes, o mesmo $\boxplus$ multiplicativo;
- a tabela de integradores da etapa (c) é o RK4 contra o RKMK do [notebook 6](../../kalman/notebooks/kalman_quadrotor_quaternions.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/kalman/notebooks/kalman_quadrotor_quaternions.ipynb)), com o colchete que separa a ordem 2 da ordem 4.

**Domínio anterior:** [os números dual-complexos](X5_dualcomplexos.ipynb) ([Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/dominios/notebooks/X5_dualcomplexos.ipynb)). **Próximo domínio:** os dual quaternions, que juntam a rotação do espaço e a translação num número só, a pose de um drone.